# Final assignment: exercise collection

**Bayesian statistics for PhD researchers · Aalborg University · 2026**

This assignment is **individual** and scored out of **100 points** with `../rubric.md` (20 points in each of five areas). It counts for 50 % of the course grade (see `../../grading.md`). Deadline: **13 December 2026 at 23:55** (strict; late submissions are not accepted). Submit **one zip file** named **`firstname_lastname.zip`** (for example `anna_jensen.zip`) on the course Moodle page, containing this notebook with outputs, an HTML or PDF export of it, and all your `.stan` files.

**Seven independent problems.** Each problem has its own everyday setting and its own data, and **no problem uses results from another**. You can do them in any order.

| Problem | Setting | Main topic |
|---|---|---|
| 1 | A basketball player's free throws | Updating, the effect of the prior, prediction |
| 2 | Phone calls to a help desk | Counts, exact answer versus Stan, diagnostics |
| 3 | An ice-cream kiosk | Regression and a predictive check |
| 4 | Thumbs-up reviews of restaurants | Partial pooling and a failing sampler |
| 5 | Passing the driving test | Logistic regression |
| 6 | Is the kitchen scale accurate? | A simulation check of the method |
| 7 | How many loaves should the bakery bake? | A decision under uncertainty |

**How to work**

- **Problems 1–4: code with blanks.** Code cells contain R code (dplyr and ggplot2) with blanks `___`. Replace every `___` with the missing piece, then run the cell. The folder `models/` has a Stan template for every model of these problems, named `...-template.stan`, with blanks `___`. Copy the template to the file name used in the code (for example `p2-calls-template.stan` → `p2-calls.stan`), fill in the blanks in the copy, and submit the completed files. Two programs, `p2-calls-supplied.stan` and `p4-reviews-centred.stan`, are complete and must not be edited.
- **Problems 5–7: no template code.** Each part says exactly what to compute and report, and which file names to use; you write the R code and the Stan programs yourself, in the empty code cells (add more cells if you like). Hints are hidden under **Hints**: try first. The course notebooks and demos show every technique you need (Days 3, 5 and 6). Use the setup cell's `fit_stan()` and `check_fit()` for every Stan fit, and `set.seed(2026)` before any simulation in R.
- Run the setup cell first. The notebook must run **from a fresh kernel, top to bottom** (Kernel › Restart Kernel and Run All Cells).
- **Answer exactly what each part asks for.** Every part says which numbers, tables or figures to report; write the numbers in the answer cell below the code. Unless a part says otherwise:
  - the seeds are already in the code (`set.seed(2026)` and `seed = 2026` in `fit_stan()`); do not change them;
  - "90 % interval" means the central interval from the 5 % to the 95 % quantile;
  - round probabilities to 3 decimals and other numbers to 2 decimals.
- For every Stan fit, report the **diagnostics** that `check_fit()` prints: the largest R-hat, the smallest bulk ESS, the number of divergent transitions and of max-treedepth hits, and the smallest E-BFMI.
- You may discuss ideas with others. The code, results and explanations you submit must be your own.
- Problem 3 uses the **loo** package. Install it once in the activated environment: `conda install -c conda-forge r-loo`.
- **All data are synthetic** (invented for teaching). Do not present them as real measurements.

## Setup

In [ ]:
set.seed(2026)
options(width = 100, repr.plot.width = 8, repr.plot.height = 5)
suppressPackageStartupMessages({library(tidyverse); library(cmdstanr); library(posterior)})
DATA_DIR <- "data"; MODEL_DIR <- "models"
stopifnot(dir.exists(DATA_DIR), dir.exists(MODEL_DIR))   # run from the exercises/ folder

# Fit a Stan program from models/ with the course defaults: 4 chains, 1000 warmup + 1000 sampling iterations.
fit_stan <- function(file, data, seed = 2026, ...) {
  m <- cmdstan_model(file.path(MODEL_DIR, file), quiet = TRUE)
  m$sample(data = data, seed = seed, chains = 4, parallel_chains = 4, refresh = 0,
           show_messages = FALSE, show_exceptions = FALSE, ...)
}
# Print a summary and the sampler diagnostics to report before the results.
check_fit <- function(fit, vars) {
  print(fit$summary(vars, "mean", "sd", ~quantile(.x, c(.05, .5, .95)), "rhat", "ess_bulk", "ess_tail"))
  d <- fit$diagnostic_summary(quiet = TRUE)
  cat("divergences:", sum(d$num_divergent), " max-treedepth hits:", sum(d$num_max_treedepth),
      " min E-BFMI:", round(min(d$ebfmi), 3), "\n")
}

## Problem 1 — Free throws with a new technique

A basketball player has changed her **free-throw technique**. Last season, with the old technique, she made **124 of 200** free throws (62 %). In training with the new technique she has made **36 of 50** (`data/p1-free-throws.csv`, one row per attempt; `made` = 1 if the shot went in). Let θ be the probability that she makes a free throw with the **new** technique. The number made is Binomial(50, θ).

Three priors for θ:
- **flat:** Beta(1, 1);
- **full:** Beta(1 + 124, 1 + 76), which treats last season's 200 attempts as if they were made with the new technique;
- **discounted:** Beta(12.4, 7.6), with the same mean as last season (0.62) but the weight of only 20 attempts.

**1a.** Complete the code. For each prior, report the posterior distribution (Beta with both parameters), the posterior mean, the 90 % interval and P(θ > 0.62). Run the figure code to plot the three posteriors.

**1b.** The coach keeps the new technique if P(θ > 0.62 | data) > 0.9. For each of the three priors, answer **yes or no**: does the coach keep the technique?

**1c.** Under the **discounted** prior, report the probability that she makes **at least 8 of her next 10** free throws: (i) exactly, (ii) by simulation, (iii) with the plug-in binomial calculation that uses the posterior mean of θ.

<details><summary><b>Hint</b> (click to open)</summary>

For a Beta(a, b) prior and y makes in n attempts, the posterior is Beta(a + y, b + n − y). P(θ > 0.62) is `1 - pbeta(0.62, a, b)`.

</details>

In [ ]:
p1 <- read.csv(file.path(DATA_DIR, "p1-free-throws.csv"))
y <- sum(p1$made); n <- nrow(p1)
c(made = y, attempts = n)

# 1a: one row per prior
post <- tibble(prior = c("flat", "full", "discounted"),
               a = c(1, 1 + 124, 12.4),
               b = c(1, 1 + 76, 7.6)) |>
  mutate(a_post = a + ___,                       # prior a + number of makes
         b_post = b + ___,                   # prior b + number of misses
         mean = a_post / (a_post + b_post),
         q05 = qbeta(0.05, a_post, b_post),
         q95 = qbeta(___, a_post, b_post),
         P_gt_062 = 1 - pbeta(___, a_post, b_post))
post

# Figure: the three posterior densities
post |>
  select(prior, a_post, b_post) |>
  cross_join(tibble(theta = seq(0.3, 0.95, length.out = 500))) |>
  mutate(density = dbeta(theta, ___, ___)) |>
  ggplot(aes(theta, density, colour = prior)) +
  geom_line(linewidth = 1) +
  geom_vline(xintercept = 0.62, linetype = 2) +
  labs(x = "theta: probability of making a free throw", y = "Posterior density", colour = "Prior")

In [ ]:
# 1c: at least 8 of the next 10, discounted prior
a <- post$a_post[3]; b <- post$b_post[3]
k <- 0:10
pred_exact <- choose(10, k) * beta(a + k, b + 10 - k) / beta(a, b)   # beta-binomial probabilities of k makes
set.seed(2026)
theta_sim <- rbeta(100000, a, b)
next10 <- rbinom(100000, 10, ___)
c(exact = sum(pred_exact[k >= ___]),
  simulated = mean(next10 >= ___),
  plug_in = 1 - pbinom(7, 10, ___))

**Your answer:** *(write here)*

## Problem 2 — Phone calls to a help desk

A university help desk counted the **phone calls** it received on 30 days (`data/p2-help-desk-calls.csv`). It is open for a different number of hours on different days (`hours_open`). Let λ be the expected number of **calls per hour**. The model is

calls on day *i* ~ Poisson(λ × hours_open on day *i*),  λ ~ Gamma(shape 36, rate 12).

**2a.** Complete the code. Report the 2.5 % and 97.5 % quantiles of the prior, the posterior distribution of λ (Gamma with both parameters), its mean, the 90 % interval and P(λ > 3.5).

**2b.** Copy `models/p2-calls-template.stan` to `models/p2-calls.stan` and fill in the blanks. Complete and run the code. Report the diagnostics, the posterior mean and 90 % interval of λ, P(λ > 3.5) and its Monte Carlo standard error (MCSE), and the difference (Stan − exact) divided by the MCSE. Show the trace plot.

**2c.** Run the supplied program `models/p2-calls-supplied.stan` (do not edit it). Report R-hat and bulk ESS for `a`, `b` and `lambda`, and the number of max-treedepth hits. Answer with one of "a", "b", "a + b": which quantity is determined by the data? Then copy `models/p2-calls-fixed-template.stan` to `models/p2-calls-fixed.stan`, fill in the blanks, run it, and report R-hat and bulk ESS for `log_lambda` and `lambda` and the number of max-treedepth hits.

**2d.** Using the 2b fit, report the probability of **more than 35 calls** on a day open 8 hours, and show the plot.

<details><summary><b>Hint</b> (click to open)</summary>

With Poisson counts yᵢ and opening hours tᵢ, a Gamma(a, b) prior gives the posterior Gamma(a + Σyᵢ, b + Σtᵢ). In Stan, `poisson(lambda * hours_open)` uses one expected count per day.

</details>

In [ ]:
p2 <- read.csv(file.path(DATA_DIR, "p2-help-desk-calls.csv"))
a0 <- 36; b0 <- 12
qgamma(c(0.025, 0.975), a0, b0)                   # prior quantiles
a_post <- a0 + ___
b_post <- b0 + ___
exact <- c(mean = a_post / b_post,
           q05 = qgamma(0.05, a_post, b_post), q95 = qgamma(0.95, a_post, b_post),
           P_gt_3.5 = 1 - pgamma(___, a_post, b_post))
c(a_post = a_post, b_post = b_post)
round(exact, 3)

In [ ]:
# 2b: uses models/p2-calls.stan (your completed copy of the template)
d2 <- list(N = nrow(p2), hours_open = p2$hours_open, calls = p2$calls, a0 = a0, b0 = b0, hours_new = ___)
fit2 <- fit_stan("p2-calls.stan", d2)
check_fit(fit2, "lambda")
dr2 <- fit2$draws(c("lambda", "calls_new"), format = "df") |> as_tibble()
ggplot(dr2, aes(.iteration, lambda, colour = factor(.chain))) +
  geom_line(alpha = 0.6) +
  labs(title = "Trace plot of lambda", x = "Iteration", y = "lambda (calls per hour)", colour = "Chain")
ind <- as.numeric(dr2$lambda > ___)
c(stan_mean = mean(dr2$lambda), stan_q05 = unname(quantile(dr2$lambda, 0.05)),
  stan_q95 = unname(quantile(dr2$lambda, 0.95)), stan_P_gt_3.5 = mean(ind), mcse = mcse_mean(ind),
  difference_in_mcse = unname((mean(ind) - exact["P_gt_3.5"]) / ___))

In [ ]:
# 2c: the supplied program, then your repaired one (models/p2-calls-fixed.stan)
d2c <- list(N = nrow(p2), hours_open = p2$hours_open, calls = p2$calls)
fit2s <- fit_stan("p2-calls-supplied.stan", d2c)
check_fit(fit2s, c("a", "b", "lambda"))
fit2f <- fit_stan(___, d2c)
check_fit(fit2f, c("log_lambda", "lambda"))

In [ ]:
# 2d: calls on a day open 8 hours
c(P_more_than_35 = mean(dr2$calls_new > ___))
ggplot(dr2, aes(calls_new)) +
  geom_bar(fill = "grey60") +
  geom_vline(xintercept = 35.5, linetype = 2) +
  labs(x = "Calls on a day open 8 hours", y = "Draws")

**Your answer:** *(write here)*

## Problem 3 — An ice-cream kiosk

An ice-cream kiosk recorded its **sales** (number of ice creams sold) on 60 days, with the day's **maximum temperature** (°C) and whether it was a **weekend** day (`data/p3-ice-cream.csv`). The code uses the centred temperature *t* = temperature − 20.

**Model N:** sales ~ Normal(α + β_temp · *t* + β_weekend · weekend, σ), with α ~ Normal(30, 15), β_temp ~ Normal(0, 3), β_weekend ~ Normal(0, 10), σ ~ Exponential(0.1).

**Model L:** log(sales) ~ Normal(α + β_temp · *t* + β_weekend · weekend, σ), with α ~ Normal(log 30, 0.5), β_temp ~ Normal(0, 0.1), β_weekend ~ Normal(0, 0.5), σ ~ Exponential(2).

**3a.** Complete the prior predictive code for model N. For a **12 °C weekday** and a **30 °C weekend day**, report the 5 %, 50 % and 95 % quantiles of the simulated sales and the proportion of negative values.

**3b.** Copy `models/p3-sales-normal-template.stan` to `models/p3-sales-normal.stan`, fill in the blanks and run the code. Report the diagnostics and the posterior mean and 90 % interval of α, β_temp, β_weekend and σ.

**3c.** Posterior predictive check for model N with *r* = SD(sales on days with temperature ≥ 22 °C) / SD(sales on days below 22 °C). Report the observed *r* and the p-value P(*r*_rep ≥ *r*_obs), and show the histogram.

**3d.** Copy `models/p3-sales-lognormal-template.stan` to `models/p3-sales-lognormal.stan`, fill in the blanks and run the code. Report the diagnostics, the posterior mean and 90 % interval of exp(β_temp) and exp(β_weekend), and the p-value of the check in 3c for model L. Report elpd_diff and se_diff from the PSIS-LOO comparison, and answer **yes or no**: is the elpd difference larger than 2 standard errors?

**3e.** For a **28 °C Saturday**, report for each model the median and 90 % interval of predicted sales and the **stock**: the 90 % quantile rounded up, so that the probability of running out is below 10 %.

In [ ]:
p3 <- read.csv(file.path(DATA_DIR, "p3-ice-cream.csv")) |> mutate(temp_c = temperature_C - 20)

# 3a: prior predictive simulation for model N
set.seed(2026)
S <- 4000
pp <- tibble(alpha = rnorm(S, 30, 15), b_temp = rnorm(S, 0, 3),
             b_weekend = rnorm(S, 0, 10), sigma = rexp(S, 0.1)) |>
  mutate(cold_weekday = rnorm(S, alpha + b_temp * (12 - 20), sigma),
         hot_weekend  = rnorm(S, alpha + b_temp * (___ - 20) + ___, sigma))
pp |>
  summarise(across(c(cold_weekday, hot_weekend),
                   list(q05 = \(x) quantile(x, 0.05), q50 = median, q95 = \(x) quantile(x, 0.95),
                        P_negative = \(x) mean(x ___)))) |>
  pivot_longer(everything()) |>
  mutate(value = round(value, 3))

In [ ]:
# 3b: model N (models/p3-sales-normal.stan); sales_new is for a 28 C Saturday
d3 <- list(N = nrow(p3), temp_c = p3$temp_c, weekend = p3$weekend, sales = p3$sales,
           M = 1, temp_new = array(___), weekend_new = array(___))
fit3n <- fit_stan("p3-sales-normal.stan", d3)
check_fit(fit3n, c("alpha", "b_temp", "b_weekend", "sigma"))

In [ ]:
# 3c: does the spread of sales grow with temperature?
r_stat <- function(s) sd(s[p3$temperature_C >= 22]) / sd(s[p3$temperature_C ___])
r_obs <- r_stat(p3$sales)
r_rep <- apply(fit3n$draws("sales_rep", format = "matrix"), 1, r_stat)
c(r_obs = r_obs, p_value = mean(r_rep >= ___))
ggplot(tibble(r_rep), aes(r_rep)) +
  geom_histogram(bins = 40, fill = "grey70", colour = "white") +
  geom_vline(xintercept = r_obs, colour = "firebrick", linewidth = 1) +
  labs(x = "SD ratio of sales (warm days / cold days), model N", y = "Replicated datasets")

In [ ]:
# 3d: model L (models/p3-sales-lognormal.stan) and LOO
fit3l <- fit_stan("p3-sales-lognormal.stan", d3)
check_fit(fit3l, c("alpha", "b_temp", "b_weekend", "sigma"))
fit3l$draws(c("b_temp", "b_weekend"), format = "df") |> as_tibble() |>
  summarise(across(c(b_temp, b_weekend),
                   list(mean = \(x) mean(exp(x)), q05 = \(x) quantile(exp(x), 0.05),
                        q95 = \(x) quantile(___, 0.95))))
r_rep_l <- apply(fit3l$draws("sales_rep", format = "matrix"), 1, r_stat)
c(p_value_model_L = mean(r_rep_l >= r_obs))
loo_n <- loo::loo(fit3n$draws("log_lik", format = "matrix"))
loo_l <- loo::loo(fit3l$draws("log_lik", format = "matrix"))
loo::loo_compare(list(model_N = loo_n, model_L = ___))

In [ ]:
# 3e: stock for a 28 C Saturday
pred3 <- bind_rows(
  tibble(model = "N", sales = as.vector(fit3n$draws("sales_new[1]", format = "matrix"))),
  tibble(model = "L", sales = as.vector(fit3l$draws("sales_new[1]", format = "matrix"))))
pred3 |>
  group_by(model) |>
  summarise(median = median(sales), q05 = quantile(sales, 0.05), q95 = quantile(sales, 0.95),
            stock = ceiling(quantile(sales, ___)))
ggplot(pred3, aes(sales, fill = model)) +
  geom_density(alpha = 0.4) +
  labs(x = "Ice creams sold on a 28 C Saturday", y = "Density", fill = "Model")

**Your answer:** *(write here)*

## Problem 4 — Thumbs-up reviews of restaurants

A review website lets customers give a restaurant a **thumbs up** or a **thumbs down**. `data/p4-restaurant-reviews.csv` has, for 15 restaurants in one town, the number of `reviews` and the number of `thumbs_up`. Let pⱼ be the probability that a review of restaurant *j* is a thumbs up, and θⱼ = logit(pⱼ).

**4a. No pooling.** Each restaurant gets its own Beta(1, 1) prior. Complete the code, report the table, and list the restaurants with rank 1 by raw proportion.

**4b. Partial pooling.** The model is θⱼ ~ Normal(μ, τ), thumbs_upⱼ ~ Binomial(reviewsⱼ, logit⁻¹(θⱼ)), μ ~ Normal(1, 1), τ ~ Normal⁺(0, 1). The supplied program `models/p4-reviews-centred.stan` (do not edit it) uses the centred form. Copy `models/p4-reviews-noncentred-template.stan` to `models/p4-reviews-noncentred.stan` and fill in the blanks (θⱼ = μ + τ · zⱼ with zⱼ ~ Normal(0, 1)). Run the code and report, for **each** of the two fits: the number of divergent transitions, the smallest E-BFMI, and R-hat and bulk ESS for τ. Report the largest τ among the divergent draws of the centred fit, and show the plot.

**4c.** Complete the code that compares no pooling (4a), partial pooling (4b, non-centred) and complete pooling (one rate for all restaurants, Beta(1, 1) prior). Report the table and the name of the restaurant whose estimate changes **least** between no pooling and partial pooling, and show the plot.

**4d.** From the non-centred fit, report: (i) the three restaurants with the highest probability of being the best, with these probabilities; (ii) the mean and 90 % interval of `p_new` (a new restaurant) and P(`p_new` > 0.8); (iii) the probability that the next review of R15 is a thumbs up, and the same probability for a new restaurant.

In [ ]:
p4 <- read.csv(file.path(DATA_DIR, "p4-restaurant-reviews.csv"))

# 4a: no pooling, Beta(1, 1) prior for each restaurant
nopool <- p4 |>
  mutate(raw = thumbs_up / reviews,
         post_mean = (thumbs_up + 1) / (reviews + ___),
         q05 = qbeta(0.05, thumbs_up + 1, reviews - thumbs_up + 1),
         q95 = qbeta(0.95, ___, ___),
         rank_raw = min_rank(desc(raw)),
         rank_post = min_rank(desc(___)))
nopool |> mutate(across(c(raw, post_mean, q05, q95), \(x) round(x, 3)))
nopool |> filter(rank_raw == 1) |> pull(restaurant)

In [ ]:
# 4b: centred (supplied) and non-centred (models/p4-reviews-noncentred.stan)
d4 <- list(J = nrow(p4), reviews = p4$reviews, thumbs_up = p4$thumbs_up)
fit4c <- fit_stan("p4-reviews-centred.stan", d4)
check_fit(fit4c, c("mu", "tau"))
fit4n <- fit_stan("p4-reviews-noncentred.stan", d4, adapt_delta = ___)
check_fit(fit4n, c("mu", "tau"))

div <- fit4c$sampler_diagnostics(format = "df") |> as_tibble() |> select(.chain, .iteration, divergent__)
dr4c <- fit4c$draws(c("tau", "theta[14]"), format = "df") |> as_tibble() |>
  left_join(div, by = c(".chain", ".iteration"))
c(divergent_draws = sum(dr4c$divergent__),
  largest_tau_among_divergent = max(dr4c$tau[dr4c$divergent__ == ___]))
ggplot(dr4c, aes(`theta[14]`, log(tau))) +
  geom_point(colour = "grey60", alpha = 0.4, size = 0.8) +
  geom_point(data = filter(dr4c, divergent__ == 1), colour = "firebrick", size = 3) +
  labs(title = "Centred model (red = divergent draws)", x = "theta for R14 (log-odds)", y = "log(tau)")

In [ ]:
# 4c: no, partial and complete pooling
partial <- colMeans(fit4n$draws("p", format = "matrix"))
complete <- (sum(p4$thumbs_up) + 1) / (sum(p4$reviews) + ___)
comp <- nopool |>
  transmute(restaurant, reviews, no_pooling = post_mean, partial = partial, complete = complete) |>
  mutate(change = partial - ___)
comp |> mutate(across(c(no_pooling, partial, complete, change), \(x) round(x, 3)))
comp |> slice_min(abs(change), n = 1)
comp |>
  pivot_longer(c(no_pooling, partial, complete), names_to = "model", values_to = "estimate") |>
  mutate(model = factor(model, c("no_pooling", "partial", "complete"))) |>
  ggplot(aes(model, estimate, group = restaurant)) +
  geom_line(colour = "grey60") +
  geom_point(aes(size = reviews)) +
  labs(x = NULL, y = "Estimated thumbs-up rate", size = "Reviews")

In [ ]:
# 4d
best <- fit4n$draws("is_best", format = "matrix")
tibble(restaurant = p4$restaurant, P_best = colMeans(best)) |> slice_max(P_best, n = ___)
p_new <- as.vector(fit4n$draws("p_new", format = "matrix"))
p15 <- as.vector(fit4n$draws("p[15]", format = "matrix"))
c(p_new_mean = mean(p_new), p_new_q05 = unname(quantile(p_new, 0.05)), p_new_q95 = unname(quantile(p_new, 0.95)),
  P_new_above_0.8 = mean(p_new > ___), next_review_R15 = mean(___), next_review_new = mean(p_new))

**Your answer:** *(write here)*

## Problem 5 — Passing the driving test

A driving school recorded, for **200 learners** who took the practical driving test, the number of driving **lessons** they had taken, their **age**, whether they had also **practised with a family member** (`family_practice`, 1 = yes), and whether they **passed** at the first attempt (`data/p5-driving-test.csv`). The code uses *l* = (lessons − 20)/10 and *a* = (age − 30)/10. The model is

passed ~ Bernoulli(logit⁻¹(α + β₁ *l* + β₂ *a* + β₃ family_practice)),  α ~ Normal(0, 1.5),  β₁, β₂, β₃ ~ Normal(0, 1).

**No template code in this problem:** write the R code and the Stan program yourself. Use `set.seed(2026)` before the prior simulation and `fit_stan()` / `check_fit()` for the fit.

**5a.** Prior predictive check. Draw 2000 values of (α, β₁, β₂, β₃) from the priors and compute the implied pass probability of each of the 200 learners for each draw. Do the same with the very wide priors Normal(0, 10) on all four coefficients. Report, for both, the proportion of implied pass probabilities below 0.01 or above 0.99, and show the two histograms.

**5b.** Write the Stan program `models/p5-driving.stan`: a logistic regression with a design matrix X (columns *l*, *a*, family_practice), the priors above, `p_new` (pass probabilities for the rows of a matrix `X_new`) and `passed_rep` (a replicated dataset) in `generated quantities`. Fit it. Report the diagnostics and the posterior mean and 90 % interval of the odds ratios exp(β₁) (10 more lessons), exp(β₂) (10 years older) and exp(β₃) (family practice).

**5c.** Report the posterior mean and 90 % interval of the pass probability for: (i) a 20-year-old with 15 lessons and no family practice; (ii) a 20-year-old with 30 lessons and no family practice; (iii) a 40-year-old with 30 lessons and family practice. Plot the posterior mean and 90 % band of P(pass) against the number of lessons (5 to 45) for a 25-year-old, with and without family practice.

**5d.** Posterior predictive check by group. Groups: lessons < 15, 15–24, 25–34 and 35 or more; and family_practice 0 and 1. For each group report the number of learners, the observed pass rate and the 90 % interval of the replicated pass rate (from `passed_rep`), as a table. For each group, answer **yes or no**: is the observed pass rate inside the 90 % interval of the replicated pass rate? Finally, answer **yes or no**: do these data alone show that practising with family **causes** a higher pass rate? (The learners were not randomly assigned to practise with family.)

<details><summary><b>Hints</b> (open only when you are stuck)</summary>

- **Data:** `p5 <- read.csv(file.path(DATA_DIR, "p5-driving-test.csv")) |> mutate(l = (lessons - 20) / 10, a = (age - 30) / 10)` and `X <- as.matrix(select(p5, l, a, family_practice))`.
- **5a:** for each prior draw, the 200 log-odds are α + X β. One way: draw α as a vector of length S and the β's as an S × 3 matrix `B`; then `plogis(alpha + B %*% t(X))` is an S × 200 matrix of probabilities. Day 6 morning, Exercise A2, shows the same idea for one predictor.
- **5b:** Day 6's `models/logistic.stan` is a logistic regression with a design matrix; here the data are called `passed` and all three β's have prior SD 1. `bernoulli_logit_rng(alpha + X * beta)` gives `passed_rep`.
- **Odds ratios:** take `exp()` of each draw of β, then the mean and quantiles.
- **5c:** put the three learners and the curve's learners (lessons 5–45, age 25, family practice 0 and 1) as rows of one `X_new`, with *l* and *a* computed the same way as for the data. Column k of `fit5$draws("p_new", format = "matrix")` holds the draws of P(pass) for row k.
- **5d:** `fit5$draws("passed_rep", format = "matrix")` has one row per draw and one column per learner. For a group, the replicated pass rate of each draw is `rowMeans()` of that group's columns. `cut(p5$lessons, c(-Inf, 14, 24, 34, Inf))` makes the lesson groups. Day 6 morning, Exercise A4, shows a grouped check.
- **Causality:** think of Day 6 afternoon: what else might differ between learners who practised with family and those who did not?

</details>

In [ ]:
# Problem 5: your code

**Your answer:** *(write here)*

## Problem 6 — Is the kitchen scale accurate? A simulation check

You want to know whether a kitchen scale reads correctly. The plan is to weigh a **500 g reference weight** *n* times and fit

yᵢ ~ Normal(μ, σ),  μ ~ Normal(500, 20),  σ ~ Exponential(0.2),

where μ is the scale's average reading for the weight (g) and σ is the reading-to-reading spread (g). Before collecting data, you check with **simulated data** whether this method recovers known values. The true values are **μ = 503 g** and **σ = 4 g**. There is no data file; the code simulates the data.

**No template code in this problem:** write the R code and the Stan program yourself.

**6a.** Write the Stan program `models/p6-scale.stan` for this model, with the prior mean and SD of μ passed as data (so that 6b can change them). Write a function that simulates one dataset of n readings from the true values, fits the model (2 chains of 500 warmup and 500 sampling iterations are enough) and returns the posterior mean and 90 % interval of μ and σ, the largest R-hat and the number of divergences. After `set.seed(2026)`, use it for one simulated dataset with n = 8: report the posterior mean and 90 % interval of μ and σ, and for each answer **yes or no**: does the interval contain the true value?

**6b.** Repeat the simulation 100 times for each of four settings: n = 8, n = 3 and n = 30 with the prior above, and n = 8 with the prior μ ~ Normal(500, 1). Report a table. It has, for each setting and parameter: the coverage (proportion of 90 % intervals containing the true value), its Monte Carlo standard error, the bias, the average interval width, the relative width (average width divided by the true value), the largest R-hat and the number of fits with divergences. Plot the coverage with ± 2 Monte Carlo standard errors for every setting and parameter, with a line at 0.9.

**6c.** From the table, answer: (i) at n = 8, which parameter has the larger relative width, μ or σ? (ii) Which setting has the lowest coverage for μ? (iii) Is that coverage below 0.5 (yes or no)?

<details><summary><b>Hints</b> (open only when you are stuck)</summary>

- **Stan program:** the model is Day 4's normal model with two unknowns; add `real prior_mu_mean; real<lower=0> prior_mu_sd;` to the data block and use them in the prior of μ. `sigma ~ exponential(0.2);`.
- **Compile once, fit many times:** `m6 <- cmdstan_model(file.path(MODEL_DIR, "p6-scale.stan"))` outside the function, then `m6$sample(...)` inside it, with `seed = sample.int(1e6, 1)` so that every fit gets its own seed from R's random numbers.
- **Many fits write many files:** `output_dir = tempfile()` (after `dir.create()`) keeps them apart; `refresh = 0, show_messages = FALSE` keeps the output short.
- **One row per parameter:** `f$summary(c("mu", "sigma"), "mean", ~quantile(.x, c(0.05, 0.95)), "rhat")` returns a small table; add columns for the truth, n and the prior SD with `mutate()`.
- **Repeating:** a `for` loop that stacks the results with `bind_rows()` is fine (as in the Day 5 extra exercises, C2).
- **Coverage and its Monte Carlo standard error:** coverage is the mean of `lower <= truth & truth <= upper`; its MCSE is `sqrt(coverage * (1 - coverage) / 100)` (Day 3). Bias is the mean of (posterior mean − truth).
- **Time:** 400 fits take a few minutes. Test with 5 repetitions first.

</details>

In [ ]:
# Problem 6: your code

**Your answer:** *(write here)*

## Problem 7 — How many loaves should the bakery bake?

A small bakery bakes **sourdough loaves** each morning. For **40 weekdays**, the staff noted the **demand**: the number of customers who wanted a loaf, including those who came after the loaves had sold out (`data/p7-bakery-demand.csv`). Baking a loaf costs **8 kr**; it sells for **25 kr**; loaves not sold are given away at closing time. If she bakes *k* loaves and demand is *D*, her profit is 25 · min(*k*, *D*) − 8 · *k* kr.

**Model P:** demand ~ Poisson(λ), λ ~ Normal⁺(60, 20).
**Model NB:** demand ~ NegBinomial2(μ, φ) (mean μ, variance μ + μ²/φ), μ ~ Normal⁺(60, 20), φ ~ Exponential(0.1).

**No template code in this problem:** write the R code and the Stan programs yourself.

**7a.** Write the Stan programs `models/p7-demand-poisson.stan` and `models/p7-demand-negbin.stan` for models P and NB. Each must also generate a replicated dataset `demand_rep` (40 days) and tomorrow's demand `demand_new`. Fit both. Report the observed mean and variance of demand; the diagnostics of both fits; the posterior mean and 90 % interval of λ, μ and φ; and, for each model, the median of the replicated variances and the p-value P(variance_rep ≥ observed variance).

**7b.** For each model and each price (20, 25 and 30 kr), compute the expected profit for every *k* from 40 to 110 loaves, averaging over the posterior predictive draws of tomorrow's demand. Report the best number of loaves *k*, the expected profit at that *k*, and the probability of selling out, P(*D* > *k*). Plot the expected profit against *k* for both models at 25 kr.

**7c.** Fill in this summary for the baker (model NB, price 25 kr), as a Markdown table with five rows, each with a number and a unit: (1) number of loaves to bake; (2) expected profit per day; (3) percentage of days on which she sells out; (4) number of loaves to bake if the price is 20 kr; (5) number of loaves to bake if the price is 30 kr.

<details><summary><b>Hints</b> (open only when you are stuck)</summary>

- **Stan programs:** Day 3's beta-binomial program and Day 5's regression programs show the blocks. The data are `int<lower=1> N; array[N] int<lower=0> demand;`. Normal⁺ means a lower bound of 0 on the parameter and a `normal(60, 20)` prior. `neg_binomial_2(mu, phi)` and `neg_binomial_2_rng(mu, phi)` are Stan's NegBinomial2; `poisson_rng(lambda)` gives one Poisson draw.
- **Replicated variance:** `fit$draws("demand_rep", format = "matrix")` has one row per draw; `apply(m, 1, var)` gives each replicated dataset's variance. The p-value is the share of replicated variances at least as large as the observed one.
- **Expected profit:** with `D <- as.vector(fit$draws("demand_new", format = "matrix"))`, the expected profit of baking *k* loaves at price *p* is `mean(p * pmin(k, D) - 8 * k)`. A grid made by `expand_grid(model, price, k = 40:110)` and a small function, or a `for` loop over *k*, both work.
- **The best k:** `group_by(model, price) |> slice_max(expected_profit, n = 1)`. P(sell out) is `mean(D > k)` for that *k*.
- **7c:** a Markdown table is written in an answer cell: `| quantity | value |`, then `|---|---|`, then one row per line.

</details>

In [ ]:
# Problem 7: your code

**Your answer:** *(write here)*